# KlaDB human development: full-depth ESM-2 with LoRA

This notebook compares three pipelines on the same locked human training and validation partitions: four-layer ProtBERT PCBert-Kla (`baseline`), MM-Kla's original token-gated head with a full 33-layer ESM-2 650M encoder adapted by LoRA (`token_gated`), and the existing architecture-aligned HybridKla implementation. Only the encoder and its adaptation change for MM-Kla; the gated fusion, 27 features, and residual classifier remain unchanged. The ESM-2 weights are general-purpose pretrained weights, not AttentionKla's Kla-trained adapters.

This is a development comparison, not an independent test. Neither the internal nor study-held-out test partition is scored. HybridKla-aligned is not an exact reproduction of the authors' training pipeline. Different backbones and optimizers mean the table compares complete pipelines, not a controlled test of architecture alone.

## 1. Configuration

Run on Kaggle with Internet access, two T4 GPUs, a `GITHUB_TOKEN` secret for the private repository, and a private dataset containing adjacent `split_audit.json` and `model_manifest.csv` files from `outputs/kladb_human_locked`. HybridKla also requires the checksum-verified official resource archive. The default `check` mode trains nothing. `pilot` checks ESM-2/LoRA memory and batch speed without scoring validation. `development` trains all three models for one seed. Save distinct notebook versions for seeds 42, 123, and 777.

In [ ]:
from pathlib import Path
RUN_MODE = 'check'  # 'check', 'pilot', or 'development'
RUN_SEED = 42
DECLARED_SEEDS = (42, 123, 777)
EXPECTED_COMMIT = None  # Pin later runs to the first run's full commit hash.
LOCK_DIR = None  # Set when more than one locked input is mounted.
LOCAL_HYBRIDKLA_ARCHIVE = None  # Absolute path to mounted official HybridKla.bin or .zip.
ROOT = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO = ROOT / 'MM-Kla'
PROJECT = REPO / 'experiment' / 'pcbert_kla_clean'
OUTPUT = ROOT / 'kla_human_esm2_lora_development'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
if RUN_MODE not in {'check', 'pilot', 'development'} or RUN_SEED not in DECLARED_SEEDS:
    raise ValueError('Invalid run mode or seed')
if not ROOT.is_dir() or not INPUT.is_dir():
    raise RuntimeError('A Kaggle runtime is required')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Mode:', RUN_MODE, 'Seed:', RUN_SEED)

## 2. Code and environment

Each run records the Git commit. A pinned commit is checked out before dependencies are installed or training begins. The private GitHub token is passed through a temporary askpass script and is not written to the notebook or output bundle.

In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, time, urllib.request, zipfile
if not REPO.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle Secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n *Username*) printf "%s\\n" x-access-token ;;\n *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        try:
            command = ['git', '-c', 'credential.helper=', 'clone', '--branch', 'main']
            if EXPECTED_COMMIT is None:
                command.extend(['--depth', '1'])
            command.extend([REPO_URL, str(REPO)])
            clone = subprocess.run(command, env=env, capture_output=True, text=True)
        finally:
            del token, env
    if clone.returncode:
        raise RuntimeError('Private clone failed; check Internet, token, and repository permissions')
elif not (REPO / '.git').is_dir():
    raise RuntimeError('The existing repository path is not a Git checkout')
if EXPECTED_COMMIT is not None:
    if not isinstance(EXPECTED_COMMIT, str) or len(EXPECTED_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in EXPECTED_COMMIT):
        raise ValueError('EXPECTED_COMMIT must be a full 40-character lowercase commit hash')
    available = subprocess.run(['git', 'cat-file', '-e', EXPECTED_COMMIT + '^{commit}'], cwd=REPO, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if available.returncode:
        raise RuntimeError('Pinned commit is unavailable; start a fresh Kaggle session')
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPO, check=True, capture_output=True, text=True)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', 'peft>=0.11,<1'], cwd=PROJECT, check=True)
help_text = subprocess.check_output([sys.executable, 'scripts/run_replication.py', '--help'], cwd=PROJECT, text=True)
if '--encoder-adaptation' not in help_text or 'token_gated' not in help_text:
    raise RuntimeError('Checkout does not contain the LoRA runner; use a fresh checkout')
hybrid_help = subprocess.check_output([sys.executable, 'scripts/run_hybridkla_architecture.py', '--help'], cwd=PROJECT, text=True)
if '--development-only' not in hybrid_help:
    raise RuntimeError('Checkout does not contain the HybridKla development runner')
import torch, transformers, peft
if RUN_MODE != 'check' and torch.cuda.device_count() < 2:
    raise RuntimeError('Training requires a 2 x T4 Kaggle accelerator')
runtime = {'git_commit': COMMIT, 'seed': RUN_SEED, 'mode': RUN_MODE, 'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2))

## 3. Locked benchmark

The attached lock and manifest must match the uncapped KlaDB human v1 benchmark. Training contains 9,970 positive and 29,910 putative-negative sites; validation contains 1,436 positive and 16,277 putative-negative sites. The internal and study-held-out partitions remain unscored.

In [ ]:
AUDIT_SHA = '2d1275577a430ff5758aa89c01a62c138c294ddc537bc67f671db3807856a0b8'
MANIFEST_SHA = '842ff1b767d60f5645d3ba9c93150ec42ed9966dcc13672a679269327a25b95e'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if LOCK_DIR is None:
    candidates = [p.parent for p in INPUT.rglob('split_audit.json') if (p.parent / 'model_manifest.csv').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one paired human lock, found {len(candidates)}; set LOCK_DIR')
    LOCK_DIR = candidates[0]
LOCK_DIR = Path(LOCK_DIR)
LOCK = LOCK_DIR / 'split_audit.json'
MANIFEST = LOCK_DIR / 'model_manifest.csv'
if sha256(LOCK) != AUDIT_SHA or sha256(MANIFEST) != MANIFEST_SHA:
    raise ValueError('Attached files differ from the locked KlaDB human v1 benchmark')
audit = json.loads(LOCK.read_text())
if audit.get('train_positive_cap') != 0 or audit.get('sampled_training_rows') != 39880 or audit.get('holdout_pmid') != '41192556':
    raise ValueError('Attached benchmark is not the uncapped human study split')
subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'human-study-check', '--human-lock', str(LOCK)], cwd=PROJECT, check=True)
print('Verified lock:', LOCK_DIR)

## 4. HybridKla resources

HybridKla-aligned uses two feature resources from the official archive. A mounted `.bin` or `.zip` is accepted if its SHA-256 matches; otherwise the official archive is downloaded. No HybridKla training rows are added to this benchmark.

In [ ]:
RESOURCE_DIR = OUTPUT / 'hybridkla_resources'
if RUN_MODE == 'development':
    archive_path = Path(LOCAL_HYBRIDKLA_ARCHIVE) if LOCAL_HYBRIDKLA_ARCHIVE else OUTPUT / 'HybridKla.zip'
    if not archive_path.is_file() and LOCAL_HYBRIDKLA_ARCHIVE is None:
        urllib.request.urlretrieve('https://transkla.zzu.edu.cn/api/download/HybridKla.zip', archive_path)
    if not archive_path.is_file() or sha256(archive_path) != '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993':
        raise ValueError('Official HybridKla archive is missing or has changed')
    RESOURCE_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for filename in ('top10.txt', 'BLOSUM62R.txt'):
            (RESOURCE_DIR / filename).write_bytes(archive.read('HybridKla/train/' + filename))
    print('Verified resources:', RESOURCE_DIR)
else:
    print('HybridKla resources not needed in this mode')

## 5. ESM-2 memory and throughput pilot

The pilot performs ten synthetic 51-residue forward/backward updates with the proposed model. It verifies LoRA gradient flow and records peak GPU memory and core-model step time. It does not train on the benchmark, evaluate validation, or estimate end-to-end epoch time (data loading and validation are excluded). A 650M-parameter encoder may exceed a free-tier runtime budget even if this pilot fits GPU memory.

In [ ]:
if RUN_MODE == 'pilot':
    sys.path.insert(0, str(PROJECT / 'src'))
    from pcbert_kla_clean.backbones import load_tokenizer
    from pcbert_kla_clean.model import TokenGatedPCBertKla
    tokenizer = load_tokenizer('facebook/esm2_t33_650M_UR50D', None)
    model = TokenGatedPCBertKla(model_name='facebook/esm2_t33_650M_UR50D', encoder_layers=33, encoder_adaptation='lora', lora_rank=8, lora_alpha=32, lora_dropout=0.3).to('cuda:1')
    trainable_encoder = [name for name, parameter in model.encoder.named_parameters() if parameter.requires_grad]
    if not trainable_encoder or any('lora_' not in name for name in trainable_encoder):
        raise RuntimeError('Expected only LoRA parameters to be trainable in ESM-2')
    batch = tokenizer(['A' * 25 + 'K' + 'A' * 25] * 4, return_tensors='pt', padding=True)
    encoded = {key: value.to('cuda:1') for key, value in batch.items()}
    features = torch.zeros(4, 27, device='cuda:1')
    labels = torch.tensor([0., 1., 0., 1.], device='cuda:1')
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-5)
    torch.cuda.reset_peak_memory_stats(1)
    started = time.monotonic()
    for _ in range(10):
        optimizer.zero_grad(set_to_none=True)
        scores = model(features=features, **encoded)
        loss = torch.nn.functional.binary_cross_entropy(scores, labels)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize(1)
    seconds_per_step = (time.monotonic() - started) / 10
    report = {'steps': 10, 'seconds_per_step': seconds_per_step, 'rough_core_training_hours_30_epochs': seconds_per_step * ((39880 + 3) // 4) * 30 / 3600, 'peak_gpu_gib': torch.cuda.max_memory_allocated(1) / 2**30, 'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad)}
    (OUTPUT / 'esm2_lora_pilot.json').write_text(json.dumps(report, indent=2) + '\n')
    print(json.dumps(report, indent=2))
    del model, optimizer, batch, encoded, features, labels, scores, loss
    torch.cuda.empty_cache()
else:
    print('Pilot skipped')

## 6. Development training

The two PyTorch pipelines run concurrently, one per GPU. PCBert-Kla retains four trainable ProtBERT layers; MM-Kla uses all 33 ESM-2 layers with LoRA adapters on query/key/value projections. Both use the same training rows, 27 features, AdamW learning rate `2e-5`, linear warmup, 30-epoch maximum, patience 5, no class weighting, and validation AUPRC selection. ESM-2 uses batch size 4 as a starting point; the pilot must confirm memory fit. HybridKla-aligned runs afterward with its existing five-fold stack and branch-specific schedule. Its optimization is not identical to the other pipelines.

This notebook retains MM-Kla's original token-gated fusion. It does not carry forward the convolution or concatenation variants. The held-out partitions remain untouched.

In [ ]:
def model_command(architecture, device, destination, seed):
    common = [sys.executable, '-u', 'scripts/run_replication.py', '--run', 'human-study-development', '--human-lock', str(LOCK), '--architecture', architecture, '--epochs', '30', '--patience', '5', '--batch-size', '4', '--device', device, '--optimizer', 'adamw', '--learning-rate', '2e-5', '--weight-decay', '0.01', '--scheduler', 'linear', '--warmup-ratio', '0.1', '--positive-weighting', 'none', '--monitor-metric', 'AUPRC', '--decision-threshold', '0.5', '--seed', str(seed), '--output-dir', str(destination)]
    if architecture == 'baseline':
        return common + ['--model-name', 'Rostlab/prot_bert', '--encoder-layers', '4']
    if architecture == 'token_gated':
        return common + ['--model-name', 'facebook/esm2_t33_650M_UR50D', '--encoder-layers', '33', '--encoder-adaptation', 'lora', '--lora-rank', '8', '--lora-alpha', '32', '--lora-dropout', '0.3']
    raise ValueError(architecture)
def hybrid_command(destination, seed):
    return [sys.executable, '-u', 'scripts/run_hybridkla_architecture.py', '--benchmark-lock', str(LOCK), '--resource-dir', str(RESOURCE_DIR), '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(seed), '--esm-model', 'facebook/esm2_t30_150M_UR50D', '--esm-epochs', '40', '--lstm-epochs', '30', '--feature-epochs', '100', '--meta-epochs', '50', '--esm-batch-size', '32', '--lstm-batch-size', '32', '--feature-batch-size', '2048', '--patience', '6', '--development-only']
def run_logged(command, destination, label):
    if destination.exists():
        raise RuntimeError(f'Prior output exists: {destination}')
    destination.mkdir(parents=True)
    log_path = destination / 'training.log'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                time.sleep(60)
                lines = log_path.read_text(errors='replace').splitlines()
                print(label, lines[-1] if lines else 'running', flush=True)
        except BaseException:
            process.terminate()
            process.wait()
            raise
    if process.returncode:
        raise RuntimeError(f'{label} failed (exit {process.returncode}):\n' + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
    print(label, 'completed', flush=True)
def run_seed(seed):
    seed_dir = OUTPUT / f'seed_{seed}'
    active = {}
    try:
        for architecture, device in (('baseline', 'cuda:0'), ('token_gated', 'cuda:1')):
            destination = seed_dir / architecture
            if destination.exists():
                raise RuntimeError(f'Prior output exists: {destination}')
            destination.mkdir(parents=True)
            log_path = destination / 'training.log'
            log = log_path.open('w')
            process = subprocess.Popen(model_command(architecture, device, destination, seed), cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
            active[architecture] = (process, log, log_path)
            print('Started', architecture, 'on', device, flush=True)
        while active:
            for architecture, (process, log, log_path) in list(active.items()):
                if process.poll() is None:
                    continue
                log.close()
                if process.returncode:
                    raise RuntimeError(f'{architecture} failed (exit {process.returncode}):\n' + '\n'.join(log_path.read_text(errors='replace').splitlines()[-30:]))
                print(architecture, 'completed', flush=True)
                del active[architecture]
            if active:
                time.sleep(60)
                for architecture, (_, _, path) in active.items():
                    lines = path.read_text(errors='replace').splitlines()
                    print(architecture, lines[-1] if lines else 'running', flush=True)
    finally:
        for process, log, _ in active.values():
            if process.poll() is None:
                process.terminate()
                process.wait()
            log.close()
    hybrid_dir = seed_dir / 'hybridkla_architecture_aligned'
    run_logged(hybrid_command(hybrid_dir, seed), hybrid_dir, f'HybridKla seed {seed}')
if RUN_MODE == 'development':
    run_seed(RUN_SEED)
else:
    print('No benchmark training in check or pilot mode')

## 7. Validation results

Metric values are read from each model's output. The checks below reject runs that changed the dataset, seed, model configuration, or development-only boundary. Validation AUPRC is the primary comparison; all standard metrics are reported. Results from a single seed are exploratory.

In [ ]:
import pandas as pd
from IPython.display import display
METRICS = ('AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC')
SETTINGS = {'mm_kla_architecture': 'token_gated', 'mm_kla_encoder': 'facebook/esm2_t33_650M_UR50D', 'mm_kla_encoder_layers': 33, 'mm_kla_encoder_adaptation': 'lora', 'lora_rank': 8, 'lora_alpha': 32, 'lora_dropout': 0.3, 'baseline_encoder': 'Rostlab/prot_bert', 'baseline_encoder_layers': 4, 'optimizer': 'adamw', 'learning_rate': 2e-5, 'weight_decay': 0.01, 'scheduler': 'linear', 'warmup_ratio': 0.1, 'epochs': 30, 'patience': 5, 'batch_size': 4, 'positive_weighting': 'none', 'hybrid_esm_model': 'facebook/esm2_t30_150M_UR50D', 'hybrid_esm_epochs': 40, 'hybrid_lstm_epochs': 30, 'hybrid_feature_epochs': 100, 'hybrid_meta_epochs': 50, 'hybrid_esm_batch_size': 32, 'hybrid_lstm_batch_size': 32, 'hybrid_feature_batch_size': 2048, 'hybrid_patience': 6, 'hybrid_folds': 5, 'selection_metric': 'AUPRC', 'threshold': 0.5}
if RUN_MODE == 'development':
    records = []
    seed_dir = OUTPUT / f'seed_{RUN_SEED}'
    for architecture, name, layers, adaptation in (('baseline', 'Rostlab/prot_bert', 4, 'full'), ('token_gated', 'facebook/esm2_t33_650M_UR50D', 33, 'lora')):
        matches = list((seed_dir / architecture).rglob('validation_metrics.json'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected one validation result for {architecture}, found {len(matches)}')
        config = json.loads(matches[0].with_name('run_config.json').read_text())
        expected = {'run': 'human-study-development', 'architecture': architecture, 'test_rows': 0, 'extra_test_rows': {}, 'model_seed': RUN_SEED, 'human_lock_sha256': AUDIT_SHA, 'model_name': name, 'encoder_layers': layers, 'encoder_adaptation': adaptation, 'optimizer': 'adamw', 'learning_rate': 2e-5, 'weight_decay': 0.01, 'scheduler': 'linear', 'warmup_ratio': 0.1, 'epochs': 30, 'patience': 5, 'batch_size': 4, 'positive_weighting': 'none', 'monitor_metric': 'AUPRC', 'decision_threshold': 0.5, 'training_rows': 39880, 'validation_rows': 17713}
        if any(config.get(key) != value for key, value in expected.items()):
            raise ValueError(f'{architecture} run crossed the development boundary or changed settings')
        if adaptation == 'lora' and any(config.get(key) != value for key, value in {'lora_rank': 8, 'lora_alpha': 32, 'lora_dropout': 0.3}.items()):
            raise ValueError('MM-Kla LoRA settings changed')
        metrics = json.loads(matches[0].read_text())
        records.append({'seed': RUN_SEED, 'architecture': architecture, 'selected_epoch': config['selected_epoch'], **{metric: metrics[metric] for metric in METRICS}})
    hybrid = json.loads((seed_dir / 'hybridkla_architecture_aligned' / 'run_report.json').read_text())
    expected_hybrid = {'esm_epochs': 40, 'lstm_epochs': 30, 'feature_epochs': 100, 'meta_epochs': 50, 'esm_batch_size': 32, 'lstm_batch_size': 32, 'feature_batch_size': 2048, 'patience': 6}
    if hybrid['mode'] != 'development-only' or hybrid['test_metrics'] or hybrid['benchmark_lock_sha256'] != AUDIT_SHA or hybrid['benchmark_manifest_sha256'] != MANIFEST_SHA or hybrid['seed'] != RUN_SEED or hybrid['folds'] != 5 or hybrid['model_name'] != SETTINGS['hybrid_esm_model'] or hybrid['selection_metric'] != 'validation AUPRC' or hybrid['threshold'] != 0.5 or any(hybrid['hyperparameters'].get(key) != value for key, value in expected_hybrid.items()):
        raise ValueError('HybridKla run crossed the development boundary or changed settings')
    records.append({'seed': RUN_SEED, 'architecture': 'hybridkla_architecture_aligned', 'selected_epoch': hybrid['fold_reports']['META']['selected_epoch'], **{metric: hybrid['validation_metrics'][metric] for metric in METRICS}})
    table = pd.DataFrame(records)
    display(table.sort_values('architecture').round(4))
    paired = table.set_index('architecture')['AUPRC']
    print('MM-Kla minus PCBert-Kla AUPRC:', round(paired['token_gated'] - paired['baseline'], 4))
    print('MM-Kla minus HybridKla-aligned AUPRC:', round(paired['token_gated'] - paired['hybridkla_architecture_aligned'], 4))
    table.to_csv(OUTPUT / 'validation_results.csv', index=False)
    summary = {'schema_version': 1, 'seed': RUN_SEED, 'git_commit': COMMIT, 'audit_sha256': AUDIT_SHA, 'manifest_sha256': MANIFEST_SHA, 'settings': SETTINGS, 'records': records}
    (OUTPUT / f'seed_{RUN_SEED}_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
else:
    print('No validation results in check or pilot mode')

## 8. Diagnostics bundle

A compact ZIP contains the seed's run metadata, validation predictions and metrics, and logs. The LoRA/full-model weights and official resource archive are excluded. Report results by seed and do not use the test partitions for model selection.

In [ ]:
if RUN_MODE == 'development':
    selected = [OUTPUT / 'runtime.json', OUTPUT / 'validation_results.csv', OUTPUT / f'seed_{RUN_SEED}_summary.json']
    for architecture in ('baseline', 'token_gated', 'hybridkla_architecture_aligned'):
        directory = OUTPUT / f'seed_{RUN_SEED}' / architecture
        selected.extend(path for path in directory.rglob('*') if path.is_file() and path.name in {'run_report.json', 'run_config.json', 'validation_metrics.json', 'validation_predictions.csv', 'training.log'})
    destination = OUTPUT / f'seed_{RUN_SEED}_development_diagnostics.zip'
    with zipfile.ZipFile(destination, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in selected:
            if not path.is_file():
                raise FileNotFoundError(path)
            archive.write(path, path.relative_to(OUTPUT))
    print('Saved:', destination, 'bytes:', destination.stat().st_size)
else:
    print('No development bundle in this mode')